# Intérpretes y compiladores · Evaluador y máquina virtual

Segunda mitad del lenguaje **Guanche**: se ejecuta el AST de dos formas. Primero con un evaluador que recorre el árbol, con entornos encadenados para los ámbitos, clausuras y errores semánticos con línea y columna; después compilándolo a *bytecode* para una máquina virtual de pila con saltos, llamadas y marcos explícitos. Entre medias, comprobaciones estáticas y plegado de constantes, un desensamblador y la comparación con lo que hacen `ast` y `dis` en CPython. Cada motor se contrasta con el otro y con Python en expresiones aleatorias y en programas de prueba (factorial, Fibonacci, FizzBuzz).

La primera celda reúne, sin cambios, el *front end* del notebook anterior: el analizador léxico, los nodos del AST, el analizador descendente recursivo, el generador de expresiones aleatorias y los programas de ejemplo.

In [1]:
import ast
import random
import re


import string
from dataclasses import dataclass

RESERVADAS = {"sea", "si", "sino", "mientras", "funcion", "devuelve", "verdadero", "falso", "nulo", "y", "o", "no"}
SIMBOLOS = ["==", "!=", "<=", ">=", "+", "-", "*", "/", "%", "^", "<", ">", "=", "(", ")", "{", "}", ",", ";"]
ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


@dataclass(frozen=True, slots=True)
class Token:
    tipo: str
    valor: object
    linea: int
    columna: int


class ErrorLexico(Exception):
    def __init__(self, mensaje, linea, columna):
        super().__init__(f"{linea}:{columna}: {mensaje}")
        self.linea, self.columna = linea, columna


def tokenizar(codigo):
    tokens = []
    i, linea, inicio_linea = 0, 1, 0
    while i < len(codigo):
        c = codigo[i]
        columna = i - inicio_linea + 1
        if c == "\n":
            linea, inicio_linea, i = linea + 1, i + 1, i + 1
        elif c in " \t\r":
            i += 1
        elif c == "#":
            while i < len(codigo) and codigo[i] != "\n":
                i += 1
        elif c in string.digits:
            j = i
            while j < len(codigo) and codigo[j] in string.digits:
                j += 1
            if j < len(codigo) and (codigo[j].isalpha() or codigo[j] == "_"):
                raise ErrorLexico(f"número mal formado {codigo[i:j + 1]!r}", linea, columna)
            tokens.append(Token("NUMERO", int(codigo[i:j]), linea, columna))
            i = j
        elif c.isalpha() or c == "_":
            j = i
            while j < len(codigo) and (codigo[j].isalnum() or codigo[j] == "_"):
                j += 1
            palabra = codigo[i:j]
            tokens.append(Token(palabra if palabra in RESERVADAS else "NOMBRE", palabra, linea, columna))
            i = j
        elif c == '"':
            j, partes = i + 1, []
            while True:
                if j >= len(codigo) or codigo[j] == "\n":
                    raise ErrorLexico("texto sin cerrar", linea, columna)
                if codigo[j] == '"':
                    break
                if codigo[j] == "\\":
                    if j + 1 < len(codigo) and codigo[j + 1] in ESCAPES:
                        partes.append(ESCAPES[codigo[j + 1]])
                        j += 2
                        continue
                    raise ErrorLexico("secuencia de escape no válida", linea, j - inicio_linea + 1)
                partes.append(codigo[j])
                j += 1
            tokens.append(Token("TEXTO", "".join(partes), linea, columna))
            i = j + 1
        else:
            for s in SIMBOLOS:
                if codigo.startswith(s, i):
                    tokens.append(Token(s, s, linea, columna))
                    i += len(s)
                    break
            else:
                raise ErrorLexico(f"carácter inesperado {c!r}", linea, columna)
    tokens.append(Token("FIN", None, linea, i - inicio_linea + 1))
    return tokens


def marcar_error(codigo, error):
    lineas = codigo.splitlines() + [""]
    print(f"  error {error}")
    print(f"    {lineas[error.linea - 1]}")
    print("    " + " " * (error.columna - 1) + "^")


FACTORIAL = """# factorial recursivo
funcion factorial(n) {
    si (n <= 1) { devuelve 1; }
    devuelve n * factorial(n - 1);
}
escribe("5! =", factorial(5));
"""
FIZZBUZZ = """sea i = 1;
mientras (i <= 15) {
    si (i % 15 == 0) { escribe("FizzBuzz"); }
    sino si (i % 3 == 0) { escribe("Fizz"); }
    sino si (i % 5 == 0) { escribe("Buzz"); }
    sino { escribe(i); }
    i = i + 1;
}
"""


from dataclasses import field, FrozenInstanceError

nodo = dataclass(frozen=True, slots=True)


def posicion():
    return field(default=(0, 0), compare=False, repr=False)


class Nodo:
    __slots__ = ()


@nodo
class Numero(Nodo):
    valor: int
    pos: tuple = posicion()

@nodo
class Texto(Nodo):
    valor: str
    pos: tuple = posicion()

@nodo
class Logico(Nodo):
    valor: bool
    pos: tuple = posicion()

@nodo
class Nulo(Nodo):
    pos: tuple = posicion()

@nodo
class Variable(Nodo):
    nombre: str
    pos: tuple = posicion()

@nodo
class Unaria(Nodo):
    op: str
    operando: Nodo
    pos: tuple = posicion()

@nodo
class Binaria(Nodo):
    op: str
    izq: Nodo
    der: Nodo
    pos: tuple = posicion()

@nodo
class Llamada(Nodo):
    funcion: Nodo
    argumentos: tuple
    pos: tuple = posicion()

@nodo
class Declaracion(Nodo):
    nombre: str
    valor: Nodo
    pos: tuple = posicion()

@nodo
class Asignacion(Nodo):
    nombre: str
    valor: Nodo
    pos: tuple = posicion()

@nodo
class Bloque(Nodo):
    sentencias: tuple
    pos: tuple = posicion()

@nodo
class Si(Nodo):
    condicion: Nodo
    entonces: Bloque
    sino: Bloque | None
    pos: tuple = posicion()

@nodo
class Mientras(Nodo):
    condicion: Nodo
    cuerpo: Bloque
    pos: tuple = posicion()

@nodo
class Funcion(Nodo):
    nombre: str
    parametros: tuple
    cuerpo: Bloque
    pos: tuple = posicion()

@nodo
class Devuelve(Nodo):
    valor: Nodo | None
    pos: tuple = posicion()

@nodo
class ExprSentencia(Nodo):
    expresion: Nodo
    pos: tuple = posicion()

@nodo
class Programa(Nodo):
    sentencias: tuple
    pos: tuple = posicion()


def como_s(n):
    match n:
        case Numero(valor=v):
            return str(v)
        case Texto(valor=v):
            return repr(v)
        case Logico(valor=v):
            return "verdadero" if v else "falso"
        case Nulo():
            return "nulo"
        case Variable(nombre=nombre):
            return nombre
        case Unaria(op=op, operando=x):
            return f"({op} {como_s(x)})"
        case Binaria(op=op, izq=a, der=b):
            return f"({op} {como_s(a)} {como_s(b)})"
        case Llamada(funcion=f, argumentos=args):
            return "(" + " ".join(["llamar", como_s(f), *map(como_s, args)]) + ")"


COMPARACIONES = ("==", "!=", "<", "<=", ">", ">=")


class ErrorSintactico(Exception):
    def __init__(self, mensaje, token):
        super().__init__(f"{token.linea}:{token.columna}: {mensaje}")
        self.linea, self.columna = token.linea, token.columna


def describir(token):
    return "el final del programa" if token.tipo == "FIN" else repr(str(token.valor))


def p(token):
    return (token.linea, token.columna)


class AnalizadorExpresiones:
    def __init__(self, codigo):
        self.tokens = tokenizar(codigo)
        self.i = 0

    @property
    def actual(self):
        return self.tokens[self.i]

    def siguiente(self):
        return self.tokens[min(self.i + 1, len(self.tokens) - 1)]

    def avanzar(self):
        t = self.tokens[self.i]
        if t.tipo != "FIN":
            self.i += 1
        return t

    def acepta(self, *tipos):
        return self.avanzar() if self.actual.tipo in tipos else None

    def espera(self, tipo, que=None):
        if self.actual.tipo != tipo:
            raise ErrorSintactico(f"se esperaba {que or repr(tipo)} y se encontró {describir(self.actual)}", self.actual)
        return self.avanzar()

    def solo_expresion(self):
        e = self.expresion()
        self.espera("FIN", "el final")
        return e

    def expresion(self):
        return self.disyuncion()

    def disyuncion(self):
        izq = self.conjuncion()
        while t := self.acepta("o"):
            izq = Binaria("o", izq, self.conjuncion(), pos=p(t))
        return izq

    def conjuncion(self):
        izq = self.negacion()
        while t := self.acepta("y"):
            izq = Binaria("y", izq, self.negacion(), pos=p(t))
        return izq

    def negacion(self):
        if t := self.acepta("no"):
            return Unaria("no", self.negacion(), pos=p(t))
        return self.comparacion()

    def comparacion(self):
        izq = self.suma()
        if t := self.acepta(*COMPARACIONES):
            izq = Binaria(t.tipo, izq, self.suma(), pos=p(t))
        return izq

    def suma(self):
        izq = self.producto()
        while t := self.acepta("+", "-"):
            izq = Binaria(t.tipo, izq, self.producto(), pos=p(t))
        return izq

    def producto(self):
        izq = self.unario()
        while t := self.acepta("*", "/", "%"):
            izq = Binaria(t.tipo, izq, self.unario(), pos=p(t))
        return izq

    def unario(self):
        if t := self.acepta("-"):
            return Unaria("-", self.unario(), pos=p(t))
        return self.potencia()

    def potencia(self):
        base = self.llamada()
        if t := self.acepta("^"):
            return Binaria("^", base, self.unario(), pos=p(t))
        return base

    def argumentos(self):
        args = []
        if self.actual.tipo != ")":
            args.append(self.expresion())
            while self.acepta(","):
                args.append(self.expresion())
        self.espera(")")
        return tuple(args)

    def llamada(self):
        e = self.primario()
        while t := self.acepta("("):
            e = Llamada(e, self.argumentos(), pos=p(t))
        return e

    def primario(self):
        t = self.actual
        match t.tipo:
            case "NUMERO":
                return Numero(self.avanzar().valor, pos=p(t))
            case "TEXTO":
                return Texto(self.avanzar().valor, pos=p(t))
            case "verdadero" | "falso":
                return Logico(self.avanzar().tipo == "verdadero", pos=p(t))
            case "nulo":
                self.avanzar()
                return Nulo(pos=p(t))
            case "NOMBRE":
                return Variable(self.avanzar().valor, pos=p(t))
            case "(":
                self.avanzar()
                e = self.expresion()
                self.espera(")")
                return e
        raise ErrorSintactico(f"se esperaba una expresión y se encontró {describir(t)}", t)


def analizar_expresion(codigo):
    return AnalizadorExpresiones(codigo).solo_expresion()


from dataclasses import fields


class AnalizadorGuanche(AnalizadorExpresiones):
    def programa(self):
        sentencias = []
        while self.actual.tipo != "FIN":
            sentencias.append(self.sentencia())
        return Programa(tuple(sentencias))

    def bloque(self):
        t = self.espera("{")
        sentencias = []
        while self.actual.tipo not in ("}", "FIN"):
            sentencias.append(self.sentencia())
        self.espera("}")
        return Bloque(tuple(sentencias), pos=p(t))

    def sentencia(self):
        t = self.actual
        match t.tipo:
            case "sea":
                self.avanzar()
                nombre = self.espera("NOMBRE", "un nombre").valor
                self.espera("=")
                valor = self.expresion()
                self.espera(";")
                return Declaracion(nombre, valor, pos=p(t))
            case "NOMBRE" if self.siguiente().tipo == "=":
                self.avanzar()
                self.avanzar()
                valor = self.expresion()
                self.espera(";")
                return Asignacion(t.valor, valor, pos=p(t))
            case "si":
                return self.si()
            case "mientras":
                self.avanzar()
                self.espera("(")
                condicion = self.expresion()
                self.espera(")")
                return Mientras(condicion, self.bloque(), pos=p(t))
            case "funcion":
                self.avanzar()
                nombre = self.espera("NOMBRE", "el nombre de la función").valor
                self.espera("(")
                parametros = []
                if self.actual.tipo != ")":
                    parametros.append(self.espera("NOMBRE", "un parámetro").valor)
                    while self.acepta(","):
                        parametros.append(self.espera("NOMBRE", "un parámetro").valor)
                self.espera(")")
                return Funcion(nombre, tuple(parametros), self.bloque(), pos=p(t))
            case "devuelve":
                self.avanzar()
                valor = None if self.actual.tipo == ";" else self.expresion()
                self.espera(";")
                return Devuelve(valor, pos=p(t))
        e = self.expresion()
        self.espera(";")
        return ExprSentencia(e, pos=p(t))

    def si(self):
        t = self.espera("si")
        self.espera("(")
        condicion = self.expresion()
        self.espera(")")
        entonces = self.bloque()
        sino = None
        if s := self.acepta("sino"):
            sino = Bloque((self.si(),), pos=p(s)) if self.actual.tipo == "si" else self.bloque()
        return Si(condicion, entonces, sino, pos=p(t))


def analizar(codigo):
    return AnalizadorGuanche(codigo).programa()


def expresion_aleatoria(rng, profundidad=3, variables=(), funciones=(), logica=False):
    def primario(p):
        opciones = ["numero"] * 3 + ["variable"] * bool(variables) * 2
        if p > 0:
            opciones += ["parentesis"] * 2 + ["llamada"] * bool(funciones)
        match rng.choice(opciones):
            case "numero":
                return str(rng.randint(0, 20))
            case "variable":
                return rng.choice(variables)
            case "parentesis":
                return f"({inicial(p - 1)})"
            case "llamada":
                args = ", ".join(inicial(p - 1) for _ in range(rng.randint(0, 2)))
                return f"{rng.choice(funciones)}({args})"

    def potencia(p):
        base = primario(p)
        if rng.random() < 0.15:
            exponente = str(rng.randint(0, 3)) + (f" ^ {rng.randint(0, 2)}" if rng.random() < 0.3 else "")
            return f"{base} ^ {exponente}"
        return base

    def unario(p):
        return ("-" if rng.random() < 0.2 else "") + potencia(p)

    def cadena(operando, operadores, p, prob):
        partes = [operando(p)]
        while rng.random() < prob and len(partes) < 7:
            partes += [rng.choice(operadores), operando(p)]
        return " ".join(partes)

    def suma(p):
        return cadena(lambda q: cadena(unario, ["*", "/", "%"], q, 0.4), ["+", "-"], p, 0.5)

    def comparacion(p):
        if rng.random() < 0.4:
            return f"{suma(p)} {rng.choice(['==', '!=', '<', '<=', '>', '>='])} {suma(p)}"
        return suma(p)

    def negacion(p):
        return ("no " if rng.random() < 0.2 else "") + comparacion(p)

    def disyuncion(p):
        return cadena(lambda q: cadena(negacion, ["y"], q, 0.3), ["o"], p, 0.3)

    inicial = disyuncion if logica else suma
    return inicial(profundidad)


def hijos(n):
    for f in fields(n):
        v = getattr(n, f.name)
        if isinstance(v, Nodo):
            yield v
        elif isinstance(v, tuple) and f.name != "pos":
            yield from (x for x in v if isinstance(x, Nodo))


def recorrer(n):
    yield n
    for h in hijos(n):
        yield from recorrer(h)


CONTADOR = """
funcion crear_contador(paso) {
    sea cuenta = 0;
    funcion siguiente() {
        cuenta = cuenta + paso;
        devuelve cuenta;
    }
    devuelve siguiente;
}
sea c = crear_contador(5);
c();
escribe(c());
"""

## 1. Evaluador por recorrido del árbol

Escribir un evaluador que recorra el AST: un `Entorno` (diccionario con enlace al entorno padre) para las variables, una función `operar` con las reglas de tipos del lenguaje (`+` suma números o une textos, `/` y `%` son enteros, las condiciones deben ser lógicas, `==` no confunde `1` con `verdadero`), las funciones nativas `escribe`, `texto` y `longitud`, y `devuelve` implementado con una excepción. `escribe` guarda las líneas en una lista para poder comparar salidas. Ejecutar el factorial y un programa con textos.

In [2]:
import operator
from dataclasses import dataclass


class ErrorGuanche(Exception):
    def __init__(self, mensaje, pos):
        super().__init__(f"{pos[0]}:{pos[1]}: {mensaje}")
        self.mensaje, self.pos = mensaje, pos
        self.linea, self.columna = pos


class Entorno:
    def __init__(self, padre=None):
        self.valores, self.padre = {}, padre

    def declarar(self, nombre, valor, pos):
        if nombre in self.valores:
            raise ErrorGuanche(f"'{nombre}' ya está declarada en este ámbito", pos)
        self.valores[nombre] = valor

    def _donde(self, nombre, pos):
        e = self
        while e is not None and nombre not in e.valores:
            e = e.padre
        if e is None:
            raise ErrorGuanche(f"variable '{nombre}' no definida", pos)
        return e

    def leer(self, nombre, pos):
        return self._donde(nombre, pos).valores[nombre]

    def asignar(self, nombre, valor, pos):
        self._donde(nombre, pos).valores[nombre] = valor


@dataclass(eq=False)
class Nativa:
    nombre: str
    funcion: object
    aridad: int | None


@dataclass(eq=False)
class Clausura:
    definicion: Funcion
    entorno: Entorno

    @property
    def nombre(self):
        return self.definicion.nombre

    @property
    def parametros(self):
        return self.definicion.parametros


def tipo(v):
    return {bool: "lógico", int: "número", str: "texto", type(None): "nulo"}.get(type(v), "función")


def mostrar(v):
    match v:
        case bool():
            return "verdadero" if v else "falso"
        case None:
            return "nulo"
        case int() | str():
            return str(v)
    return f"<funcion {v.nombre}>"


ARITMETICA = {"-": operator.sub, "*": operator.mul, "/": operator.floordiv, "%": operator.mod, "^": operator.pow}
ORDEN = {"<": operator.lt, "<=": operator.le, ">": operator.gt, ">=": operator.ge}


def igual(a, b):
    if type(a) is not type(b):
        return False
    return a == b if type(a) in (int, str, bool, type(None)) else a is b


def operar(op, a, b, pos):
    match op:
        case "==":
            return igual(a, b)
        case "!=":
            return not igual(a, b)
        case "+" if type(a) is type(b) and type(a) in (int, str):
            return a + b
        case "<" | "<=" | ">" | ">=" if type(a) is type(b) and type(a) in (int, str):
            return ORDEN[op](a, b)
        case "-" | "*" | "/" | "%" | "^" if type(a) is int and type(b) is int:
            if op in "/%" and b == 0:
                raise ErrorGuanche("división entre cero", pos)
            if op == "^" and b < 0:
                raise ErrorGuanche("exponente negativo", pos)
            return ARITMETICA[op](a, b)
    raise ErrorGuanche(f"no se puede aplicar '{op}' a {tipo(a)} y {tipo(b)}", pos)


def logico(v, pos):
    if type(v) is not bool:
        raise ErrorGuanche(f"se esperaba un valor lógico y se obtuvo {tipo(v)}", pos)
    return v


def unaria(op, v, pos):
    if op == "no":
        return not logico(v, pos)
    if type(v) is not int:
        raise ErrorGuanche(f"no se puede aplicar '-' a {tipo(v)}", pos)
    return -v


def nativas(salida):
    def longitud(t):
        if type(t) is not str:
            raise TypeError(f"longitud espera un texto y recibió {tipo(t)}")
        return len(t)

    return {
        "escribe": Nativa("escribe", lambda *args: salida.append(" ".join(map(mostrar, args))), None),
        "texto": Nativa("texto", mostrar, 1),
        "longitud": Nativa("longitud", longitud, 1),
    }


def argumentos(n):
    return f"{n} argumento{'s' * (n != 1)}"


def comprobar_llamada(funcion, n, pos):
    if not isinstance(funcion, (Nativa, Clausura)) and not hasattr(funcion, "codigo"):
        raise ErrorGuanche(f"no se puede llamar a un valor de tipo {tipo(funcion)}", pos)
    esperados = funcion.aridad if isinstance(funcion, Nativa) else len(funcion.parametros)
    if esperados is not None and esperados != n:
        raise ErrorGuanche(f"'{funcion.nombre}' espera {argumentos(esperados)} y recibió {n}", pos)


def llamar_nativa(funcion, args, pos):
    try:
        return funcion.funcion(*args)
    except TypeError as e:
        raise ErrorGuanche(str(e), pos) from None


class Retorno(Exception):
    def __init__(self, valor, pos):
        self.valor, self.pos = valor, pos


class Evaluador:
    def __init__(self):
        self.salida = []
        self.globales = Entorno()
        self.globales.valores.update(nativas(self.salida))
        self.pasos = 0

    def ejecutar(self, programa):
        try:
            for s in programa.sentencias:
                self.sentencia(s, self.globales)
        except Retorno as r:
            raise ErrorGuanche("devuelve fuera de una función", r.pos) from None
        return self.salida

    def bloque(self, bloque, entorno):
        local = Entorno(entorno)
        for s in bloque.sentencias:
            self.sentencia(s, local)

    def sentencia(self, s, ent):
        self.pasos += 1
        match s:
            case Declaracion(nombre=nombre, valor=v):
                ent.declarar(nombre, self.evaluar(v, ent), s.pos)
            case Asignacion(nombre=nombre, valor=v):
                ent.asignar(nombre, self.evaluar(v, ent), s.pos)
            case ExprSentencia(expresion=e):
                self.evaluar(e, ent)
            case Bloque():
                self.bloque(s, ent)
            case Si(condicion=c, entonces=a, sino=b):
                if logico(self.evaluar(c, ent), c.pos):
                    self.bloque(a, ent)
                elif b is not None:
                    self.bloque(b, ent)
            case Mientras(condicion=c, cuerpo=cuerpo):
                while logico(self.evaluar(c, ent), c.pos):
                    self.bloque(cuerpo, ent)
            case Funcion(nombre=nombre):
                ent.declarar(nombre, Clausura(s, ent), s.pos)
            case Devuelve(valor=v):
                raise Retorno(None if v is None else self.evaluar(v, ent), s.pos)

    def evaluar(self, n, ent):
        self.pasos += 1
        match n:
            case Numero(valor=v) | Texto(valor=v) | Logico(valor=v):
                return v
            case Nulo():
                return None
            case Variable(nombre=nombre):
                return ent.leer(nombre, n.pos)
            case Unaria(op=op, operando=x):
                return unaria(op, self.evaluar(x, ent), n.pos)
            case Binaria(op="y" | "o" as op, izq=a, der=b):
                izq = logico(self.evaluar(a, ent), n.pos)
                if izq == (op == "o"):
                    return izq
                return logico(self.evaluar(b, ent), n.pos)
            case Binaria(op=op, izq=a, der=b):
                return operar(op, self.evaluar(a, ent), self.evaluar(b, ent), n.pos)
            case Llamada(funcion=f, argumentos=args):
                funcion = self.evaluar(f, ent)
                valores = [self.evaluar(a, ent) for a in args]
                return self.llamar(funcion, valores, n.pos)

    def llamar(self, funcion, args, pos):
        comprobar_llamada(funcion, len(args), pos)
        if isinstance(funcion, Nativa):
            return llamar_nativa(funcion, args, pos)
        local = Entorno(funcion.entorno)
        local.valores.update(zip(funcion.parametros, args))
        try:
            for s in funcion.definicion.cuerpo.sentencias:
                self.sentencia(s, local)
        except Retorno as r:
            return r.valor
        return None


def ejecutar_arbol(codigo):
    return Evaluador().ejecutar(analizar(codigo))


print("\n".join(ejecutar_arbol(FACTORIAL)))
SALUDO = """
sea isla = "La Gomera";
sea letras = longitud(isla);
escribe(isla + " tiene " + texto(letras) + " caracteres");
escribe(1 == verdadero, "a" < "b", 7 / 2, -7 / 2, -7 % 3, 2 ^ 10, nulo);
"""
salida = ejecutar_arbol(SALUDO)
print("\n".join(salida))
assert salida == ["La Gomera tiene 9 caracteres", "falso verdadero 3 -4 2 1024 nulo"]
assert ejecutar_arbol(FACTORIAL) == ["5! = 120"]

5! = 120
La Gomera tiene 9 caracteres
falso verdadero 3 -4 2 1024 nulo


## 2. Entornos, ámbitos y clausuras

Comprobar con programas pequeños las reglas de ámbito: un bloque crea un entorno nuevo, una declaración dentro de él oculta a la de fuera sin modificarla y deja de existir al cerrar el bloque, una asignación modifica la variable visible más cercana, y una función recuerda el entorno donde se definió (clausura), de modo que dos contadores creados por `crear_contador` llevan cuentas independientes.

In [3]:
AMBITOS = """
sea x = "global";
si (verdadero) {
    sea x = "bloque";
    escribe("dentro:", x);
    sea solo_aqui = 1;
}
escribe("fuera:", x);
sea z = 0;
mientras (z < 3) { z = z + 1; }
escribe("z tras el bucle:", z);
"""
salida = ejecutar_arbol(AMBITOS)
print("\n".join(salida))
assert salida == ["dentro: bloque", "fuera: global", "z tras el bucle: 3"]

try:
    ejecutar_arbol(AMBITOS + "escribe(solo_aqui);")
except ErrorGuanche as e:
    print("Fuera del bloque:", e)

DOS_CONTADORES = CONTADOR + """
sea d = crear_contador(100);
escribe(d(), c(), d(), c());
"""
salida = ejecutar_arbol(DOS_CONTADORES)
print("\n".join(salida))
assert salida == ["10", "100 15 200 20"]

dentro: bloque
fuera: global
z tras el bucle: 3
Fuera del bloque: 12:9: variable 'solo_aqui' no definida
10
100 15 200 20


## 3. Errores semánticos en ejecución

Provocar los errores que el analizador sintáctico no puede ver: variable sin definir, asignación a una variable no declarada, redeclaración en el mismo ámbito, número de argumentos incorrecto, llamar a algo que no es función, tipos incompatibles, división entre cero, condición no lógica y `devuelve` fuera de una función. Mostrarlos con `marcar_error` y comprobar la posición de cada uno.

In [4]:
ERRONEOS = [
    ("sea x = 1;\nescribe(w + x);", (2, 9)),
    ("total = 5;", (1, 1)),
    ("sea a = 1;\nsea a = 2;", (2, 1)),
    ("funcion doble(n) { devuelve 2 * n; }\nescribe(doble(1, 2));", (2, 14)),
    ("sea f = 3;\nf(1);", (2, 2)),
    ('escribe("Hola" + 1);', (1, 16)),
    ("sea x = 10 / (5 - 5);", (1, 12)),
    ("si (1) { }", (1, 5)),
    ("devuelve 3;", (1, 1)),
    ("si (verdadero) { sea dentro = 1; }\nescribe(dentro);", (2, 9)),
]
errores_ejecucion = []
for fuente, esperado in ERRONEOS:
    try:
        ejecutar_arbol(fuente)
    except ErrorGuanche as e:
        marcar_error(fuente, e)
        errores_ejecucion.append((e.pos, e.mensaje))
        assert e.pos == esperado, (fuente, e.pos)
    else:
        raise AssertionError(fuente)

  error 2:9: variable 'w' no definida
    escribe(w + x);
            ^
  error 1:1: variable 'total' no definida
    total = 5;
    ^
  error 2:1: 'a' ya está declarada en este ámbito
    sea a = 2;
    ^
  error 2:14: 'doble' espera 1 argumento y recibió 2
    escribe(doble(1, 2));
                 ^
  error 2:2: no se puede llamar a un valor de tipo número
    f(1);
     ^
  error 1:16: no se puede aplicar '+' a texto y número
    escribe("Hola" + 1);
                   ^
  error 1:12: división entre cero
    sea x = 10 / (5 - 5);
               ^
  error 1:5: se esperaba un valor lógico y se obtuvo número
    si (1) { }
        ^
  error 1:1: devuelve fuera de una función
    devuelve 3;
    ^
  error 2:9: variable 'dentro' no definida
    escribe(dentro);
            ^


## 4. Comprobaciones estáticas antes de ejecutar

Escribir `comprobar(programa)`, que recorra el AST sin ejecutarlo con una pila de ámbitos y devuelva todos los errores que se pueden detectar antes de ejecutar: nombres sin declarar, redeclaraciones, `devuelve` fuera de una función y llamadas directas a una función con un número de argumentos distinto de sus parámetros. Los cuerpos de las funciones se comprueban al cerrar el ámbito que las contiene, porque pueden usar nombres declarados después (recursión mutua). Comprobar que encuentra el mismo error que la ejecución en los casos que le corresponden, que no da falsos avisos en los programas correctos y que detecta errores en ramas que la ejecución nunca visita.

In [5]:
def comprobar(programa):
    errores = []

    def buscar(ambitos, nombre):
        return next((a for a in reversed(ambitos) if nombre in a), None)

    def expresion(e, ambitos):
        match e:
            case Variable(nombre=nombre) if buscar(ambitos, nombre) is None:
                errores.append((e.pos, f"variable '{nombre}' no definida"))
            case Llamada(funcion=Variable(nombre=nombre), argumentos=args):
                ambito = buscar(ambitos, nombre)
                aridad = ambito[nombre] if ambito else None
                if aridad is not None and aridad != len(args):
                    errores.append((e.pos, f"'{nombre}' espera {argumentos(aridad)} y recibió {len(args)}"))
        for h in hijos(e):
            expresion(h, ambitos)

    def bloque(sentencias, ambitos, en_funcion, inicial=None):
        ambito = dict(inicial or {})
        ambitos = ambitos + [ambito]
        funciones = []

        def declarar(nombre, aridad, pos):
            if nombre in ambito:
                errores.append((pos, f"'{nombre}' ya está declarada en este ámbito"))
            ambito[nombre] = aridad

        for s in sentencias:
            match s:
                case Declaracion(nombre=nombre, valor=v):
                    expresion(v, ambitos)
                    declarar(nombre, None, s.pos)
                case Asignacion(nombre=nombre, valor=v):
                    expresion(v, ambitos)
                    if buscar(ambitos, nombre) is None:
                        errores.append((s.pos, f"variable '{nombre}' no definida"))
                case ExprSentencia(expresion=e):
                    expresion(e, ambitos)
                case Si(condicion=c, entonces=a, sino=b):
                    expresion(c, ambitos)
                    bloque(a.sentencias, ambitos, en_funcion)
                    if b is not None:
                        bloque(b.sentencias, ambitos, en_funcion)
                case Mientras(condicion=c, cuerpo=cuerpo):
                    expresion(c, ambitos)
                    bloque(cuerpo.sentencias, ambitos, en_funcion)
                case Bloque(sentencias=ss):
                    bloque(ss, ambitos, en_funcion)
                case Funcion(nombre=nombre, parametros=ps):
                    declarar(nombre, len(ps), s.pos)
                    funciones.append(s)
                case Devuelve(valor=v):
                    if v is not None:
                        expresion(v, ambitos)
                    if not en_funcion:
                        errores.append((s.pos, "devuelve fuera de una función"))
        for f in funciones:
            bloque(f.cuerpo.sentencias, ambitos, True, dict.fromkeys(f.parametros))

    aridades_nativas = {nombre: f.aridad for nombre, f in nativas([]).items()}
    bloque(programa.sentencias, [], False, aridades_nativas)
    return sorted(errores)


detectables = 0
for (fuente, _), error in zip(ERRONEOS, errores_ejecucion):
    estaticos = comprobar(analizar(fuente))
    print(f"{'detectado ' if error in estaticos else 'solo al ejecutar'}  {error[1]}")
    detectables += error in estaticos
assert detectables == 6

MUTUA = """
funcion es_par(n) { si (n == 0) { devuelve verdadero; } devuelve es_impar(n - 1); }
funcion es_impar(n) { si (n == 0) { devuelve falso; } devuelve es_par(n - 1); }
escribe(es_par(10), es_impar(7));
"""
for fuente in (FACTORIAL, FIZZBUZZ, CONTADOR, DOS_CONTADORES, AMBITOS, SALUDO, MUTUA):
    assert comprobar(analizar(fuente)) == []
print("Recursión mutua:", ejecutar_arbol(MUTUA))

DORMIDO = """
sea modo = "normal";
si (modo == "depuración") { escribe(contdor); }
funcion area(b, h) { devuelve b * h / 2; }
si (falso) { escribe(area(3)); }
escribe("fin");
"""
print("La ejecución no se queja:", ejecutar_arbol(DORMIDO))
for pos, mensaje in comprobar(analizar(DORMIDO)):
    print(f"  {pos[0]}:{pos[1]}: {mensaje}")
assert len(comprobar(analizar(DORMIDO))) == 2

detectado   variable 'w' no definida
detectado   variable 'total' no definida
detectado   'a' ya está declarada en este ámbito
detectado   'doble' espera 1 argumento y recibió 2
solo al ejecutar  no se puede llamar a un valor de tipo número
solo al ejecutar  no se puede aplicar '+' a texto y número
solo al ejecutar  división entre cero
solo al ejecutar  se esperaba un valor lógico y se obtuvo número
detectado   devuelve fuera de una función
detectado   variable 'dentro' no definida
Recursión mutua: ['verdadero verdadero']
La ejecución no se queja: ['fin']
  3:37: variable 'contdor' no definida
  5:26: 'area' espera 2 argumentos y recibió 1


## 5. Contrastar el evaluador con Python

Las reglas aritméticas de Guanche coinciden con las de Python para enteros (`/` es `//`, `%` sigue el signo del divisor y `^` es `**`). Evaluar 2000 expresiones aleatorias con las variables `a` y `b` y comprobar que dan lo mismo que `eval` sobre la traducción a Python, incluida la división entre cero, que debe fallar en ambos.

In [6]:
def a_python(expresion):
    return expresion.replace("^", "**").replace("/", "//")


def evaluar_texto(expresion, variables, evaluador=None):
    evaluador = evaluador or Evaluador()
    entorno = Entorno(evaluador.globales)
    entorno.valores.update(variables)
    return evaluador.evaluar(analizar_expresion(expresion), entorno)


rng = random.Random(5)
divisiones_entre_cero = 0
for _ in range(2000):
    e = expresion_aleatoria(rng, profundidad=3, variables=("a", "b"))
    variables = {"a": rng.randint(-9, 9), "b": rng.randint(-9, 9)}
    try:
        esperado = eval(a_python(e), {}, variables)
    except ZeroDivisionError:
        esperado = "error"
    try:
        obtenido = evaluar_texto(e, variables)
    except ErrorGuanche as error:
        assert error.mensaje == "división entre cero"
        obtenido = "error"
        divisiones_entre_cero += 1
    assert obtenido == esperado, (e, variables, obtenido, esperado)
print(f"2000 expresiones iguales que en Python ({divisiones_entre_cero} con división entre cero en ambos)")

2000 expresiones iguales que en Python (237 con división entre cero en ambos)


## 6. Plegado de constantes

Escribir `plegar(nodo)`, una optimización que calcula en tiempo de compilación las operaciones cuyos operandos son literales (`60 * 60 * 24` → `86400`, `"Gran " + "Canaria"`), simplifica `falso y x` y `verdadero o x`, sustituye un `si` con condición literal por la rama elegida y elimina los `mientras (falso)`. Lo que fallaría al ejecutarse (como `1 / 0`) se deja sin plegar para que el error siga ocurriendo en su sitio. Comprobar con 2000 expresiones aleatorias que el resultado no cambia y medir cuántos nodos se ahorran.

In [7]:
LITERALES = (Numero, Texto, Logico, Nulo)


def literal(valor, pos):
    match valor:
        case bool():
            return Logico(valor, pos=pos)
        case int():
            return Numero(valor, pos=pos)
        case str():
            return Texto(valor, pos=pos)
    return Nulo(pos=pos)


def valor_de(n):
    return None if isinstance(n, Nulo) else n.valor


def plegar(n):
    match n:
        case Unaria(op=op, operando=x):
            x = plegar(x)
            if isinstance(x, LITERALES):
                try:
                    return literal(unaria(op, valor_de(x), n.pos), n.pos)
                except ErrorGuanche:
                    pass
            return Unaria(op, x, pos=n.pos)
        case Binaria(op=op, izq=a, der=b):
            a, b = plegar(a), plegar(b)
            if op in ("y", "o") and isinstance(a, Logico) and a.valor == (op == "o"):
                return a
            if isinstance(a, LITERALES) and isinstance(b, LITERALES):
                try:
                    if op in ("y", "o"):
                        logico(valor_de(a), n.pos)
                        return literal(logico(valor_de(b), n.pos), n.pos)
                    return literal(operar(op, valor_de(a), valor_de(b), n.pos), n.pos)
                except ErrorGuanche:
                    pass
            return Binaria(op, a, b, pos=n.pos)
        case Llamada(funcion=f, argumentos=args):
            return Llamada(plegar(f), tuple(map(plegar, args)), pos=n.pos)
        case Declaracion(nombre=nombre, valor=v):
            return Declaracion(nombre, plegar(v), pos=n.pos)
        case Asignacion(nombre=nombre, valor=v):
            return Asignacion(nombre, plegar(v), pos=n.pos)
        case ExprSentencia(expresion=e):
            return ExprSentencia(plegar(e), pos=n.pos)
        case Devuelve(valor=v):
            return Devuelve(None if v is None else plegar(v), pos=n.pos)
        case Si(condicion=c, entonces=a, sino=b):
            c = plegar(c)
            if isinstance(c, Logico):
                elegido = a if c.valor else b
                return None if elegido is None else plegar(elegido)
            return Si(c, plegar(a), None if b is None else plegar(b), pos=n.pos)
        case Mientras(condicion=c, cuerpo=cuerpo):
            c = plegar(c)
            if isinstance(c, Logico) and not c.valor:
                return None
            return Mientras(c, plegar(cuerpo), pos=n.pos)
        case Funcion(nombre=nombre, parametros=ps, cuerpo=cuerpo):
            return Funcion(nombre, ps, plegar(cuerpo), pos=n.pos)
        case Bloque(sentencias=ss) | Programa(sentencias=ss):
            plegadas = tuple(x for x in map(plegar, ss) if x is not None)
            return type(n)(plegadas, pos=n.pos)
    return n


def contar_nodos(n):
    return sum(1 for _ in recorrer(n))


OPTIMIZABLE = """
sea segundos_dia = 60 * 60 * 24;
sea isla = "Gran " + "Canaria";
sea depurar = falso;
si (depurar y segundos_dia > 0) { escribe("depurando"); }
mientras (falso) { escribe("nunca"); }
si (no depurar o 1 / 0 == 0) { escribe(isla, segundos_dia * (2 + 5)); }
escribe(1 / 0 == 0 o verdadero);
"""
def resumen(s):
    match s:
        case Declaracion(nombre=nombre, valor=v):
            return f"sea {nombre} = {como_s(v)}"
        case ExprSentencia(expresion=e):
            return como_s(e)
        case Si(condicion=c, sino=b):
            return f"si {como_s(c)} {{…}}" + (" sino {…}" if b else "")
        case Mientras(condicion=c):
            return f"mientras {como_s(c)} {{…}}"
        case Bloque(sentencias=ss):
            return f"bloque con {len(ss)} sentencias"
    return type(s).__name__


arbol = analizar(OPTIMIZABLE)
plegado = plegar(arbol)
print("Original:")
for s in arbol.sentencias:
    print("  " + resumen(s))
print("Plegado:")
for s in plegado.sentencias:
    print("  " + resumen(s))
print(f"Nodos: {contar_nodos(arbol)} → {contar_nodos(plegado)}")

ev = Evaluador()
ev.ejecutar(Programa(plegado.sentencias[:-1]))
print("Salida del programa plegado sin la última línea:", ev.salida)
try:
    Evaluador().ejecutar(plegado)
except ErrorGuanche as e:
    print("La última línea sigue fallando donde debe:", e)
    assert e.pos == (8, 11)

rng = random.Random(6)
antes = despues = 0
for _ in range(2000):
    e = analizar_expresion(expresion_aleatoria(rng, profundidad=3, variables=("a", "b"), logica=rng.random() < 0.5))
    variables = {"a": rng.randint(-9, 9), "b": rng.randint(-9, 9)}
    resultados = []
    for version in (e, plegar(e)):
        entorno = Entorno()
        entorno.valores.update(variables)
        try:
            resultados.append(Evaluador().evaluar(version, entorno))
        except ErrorGuanche as error:
            resultados.append(("error", error.mensaje, error.pos))
    assert resultados[0] == resultados[1] and type(resultados[0]) is type(resultados[1]), e
    antes += contar_nodos(e)
    despues += contar_nodos(plegar(e))
print(f"2000 expresiones: mismo resultado y {antes} → {despues} nodos ({1 - despues / antes:.0%} menos)")

Original:
  sea segundos_dia = (* (* 60 60) 24)
  sea isla = (+ 'Gran ' 'Canaria')
  sea depurar = falso
  si (y depurar (> segundos_dia 0)) {…}
  mientras falso {…}
  si (o (no depurar) (== (/ 1 0) 0)) {…}
  (llamar escribe (o (== (/ 1 0) 0) verdadero))
Plegado:
  sea segundos_dia = 86400
  sea isla = 'Gran Canaria'
  sea depurar = falso
  si (y depurar (> segundos_dia 0)) {…}
  si (o (no depurar) (== (/ 1 0) 0)) {…}
  (llamar escribe (o (== (/ 1 0) 0) verdadero))
Nodos: 60 → 45
Salida del programa plegado sin la última línea: ['Gran Canaria 604800']
La última línea sigue fallando donde debe: 8:11: división entre cero


2000 expresiones: mismo resultado y 472292 → 344039 nodos (27% menos)


## 7. Juego de instrucciones y compilación de expresiones

Definir el juego de instrucciones de una máquina virtual de pila (`CONST`, `CARGA`, operaciones aritméticas y de comparación, `NEG`, `NO`…) y un `Codigo` con la lista de instrucciones `(operación, argumento)`, la tabla de constantes y la posición en el fuente de cada instrucción. Escribir un `Compilador` que traduzca una expresión en orden posfijo: primero los operandos y después la operación. Mostrar el código de `1 + 2 * x`, que es su notación polaca inversa.

In [8]:
from enum import IntEnum, auto


class Op(IntEnum):
    CONST = auto()
    CARGA = auto()
    DECLARA = auto()
    GUARDA = auto()
    SACA = auto()
    NEG = auto()
    NO = auto()
    SUMA = auto()
    RESTA = auto()
    MULT = auto()
    DIV = auto()
    MOD = auto()
    POT = auto()
    IGUAL = auto()
    DISTINTO = auto()
    MENOR = auto()
    MENOR_IGUAL = auto()
    MAYOR = auto()
    MAYOR_IGUAL = auto()
    SALTA = auto()
    SALTA_SI_FALSO = auto()
    Y_SALTA = auto()
    O_SALTA = auto()
    LOGICO = auto()
    ENTRA_BLOQUE = auto()
    SALE_BLOQUE = auto()
    CREA_FUNCION = auto()
    LLAMA = auto()
    DEVUELVE = auto()
    FIN = auto()


BINARIAS = {"+": Op.SUMA, "-": Op.RESTA, "*": Op.MULT, "/": Op.DIV, "%": Op.MOD, "^": Op.POT,
            "==": Op.IGUAL, "!=": Op.DISTINTO, "<": Op.MENOR, "<=": Op.MENOR_IGUAL, ">": Op.MAYOR, ">=": Op.MAYOR_IGUAL}
SIMBOLO = {op: s for s, op in BINARIAS.items()}
SALTOS = {Op.SALTA, Op.SALTA_SI_FALSO, Op.Y_SALTA, Op.O_SALTA}


@dataclass(eq=False)
class Codigo:
    nombre: str
    parametros: tuple
    instrucciones: list
    constantes: list
    posiciones: list


class Compilador:
    def __init__(self, nombre="<programa>", parametros=()):
        self.codigo = Codigo(nombre, parametros, [], [], [])

    def emite(self, op, arg=None, pos=(0, 0)):
        self.codigo.instrucciones.append((op, arg))
        self.codigo.posiciones.append(pos)
        return len(self.codigo.instrucciones) - 1

    def constante(self, valor):
        for i, c in enumerate(self.codigo.constantes):
            if type(c) is type(valor) and c == valor:
                return i
        self.codigo.constantes.append(valor)
        return len(self.codigo.constantes) - 1

    def expresion(self, n):
        match n:
            case Numero(valor=v) | Texto(valor=v) | Logico(valor=v):
                self.emite(Op.CONST, self.constante(v), n.pos)
            case Nulo():
                self.emite(Op.CONST, self.constante(None), n.pos)
            case Variable(nombre=nombre):
                self.emite(Op.CARGA, nombre, n.pos)
            case Unaria(op=op, operando=x):
                self.expresion(x)
                self.emite(Op.NO if op == "no" else Op.NEG, None, n.pos)
            case Binaria(op=op, izq=a, der=b) if op in BINARIAS:
                self.expresion(a)
                self.expresion(b)
                self.emite(BINARIAS[op], None, n.pos)


def compilar_expresion(n):
    c = Compilador("<expresión>")
    c.expresion(n)
    c.emite(Op.FIN)
    return c.codigo


codigo = compilar_expresion(analizar_expresion("1 + 2 * x"))
for (op, arg), pos in zip(codigo.instrucciones, codigo.posiciones):
    print(f"{pos[0]}:{pos[1]:<3} {op.name:<6} {'' if arg is None else arg}")
print("Constantes:", codigo.constantes)
posfija = [str(codigo.constantes[a]) if op == Op.CONST else a if op == Op.CARGA else SIMBOLO[op]
           for op, a in codigo.instrucciones[:-1]]
print("Notación polaca inversa:", " ".join(posfija))
assert posfija == ["1", "2", "x", "*", "+"]

1:1   CONST  0
1:5   CONST  1
1:9   CARGA  x
1:7   MULT   
1:3   SUMA   
0:0   FIN    
Constantes: [1, 2]
Notación polaca inversa: 1 2 x * +


## 8. Compilación de sentencias: saltos y funciones

Completar el compilador con las sentencias. Las condiciones y bucles se traducen a saltos (`SALTA`, `SALTA_SI_FALSO`) cuyo destino se rellena cuando se conoce (*backpatching*); `y` y `o` usan saltos que conservan el valor en la pila para el cortocircuito; los bloques abren y cierran un entorno (`ENTRA_BLOQUE`, `SALE_BLOQUE`); y cada función se compila aparte en su propio `Codigo`, que se guarda como constante y se convierte en clausura en ejecución con `CREA_FUNCION`. Compilar el FizzBuzz y comprobar que todos los saltos apuntan dentro del código.

In [9]:
class CompiladorGuanche(Compilador):
    def expresion(self, n):
        match n:
            case Binaria(op="y" | "o" as op, izq=a, der=b):
                self.expresion(a)
                salto = self.emite(Op.Y_SALTA if op == "y" else Op.O_SALTA, None, n.pos)
                self.expresion(b)
                self.emite(Op.LOGICO, None, n.pos)
                self.parchea(salto)
            case Llamada(funcion=f, argumentos=args):
                self.expresion(f)
                for a in args:
                    self.expresion(a)
                self.emite(Op.LLAMA, len(args), n.pos)
            case _:
                super().expresion(n)

    def parchea(self, i):
        op, _ = self.codigo.instrucciones[i]
        self.codigo.instrucciones[i] = (op, len(self.codigo.instrucciones))

    def sentencia(self, s):
        match s:
            case Declaracion(nombre=nombre, valor=v):
                self.expresion(v)
                self.emite(Op.DECLARA, nombre, s.pos)
            case Asignacion(nombre=nombre, valor=v):
                self.expresion(v)
                self.emite(Op.GUARDA, nombre, s.pos)
            case ExprSentencia(expresion=e):
                self.expresion(e)
                self.emite(Op.SACA, None, s.pos)
            case Bloque(sentencias=ss):
                self.emite(Op.ENTRA_BLOQUE, None, s.pos)
                for x in ss:
                    self.sentencia(x)
                self.emite(Op.SALE_BLOQUE, None, s.pos)
            case Si(condicion=c, entonces=a, sino=b):
                self.expresion(c)
                si_falso = self.emite(Op.SALTA_SI_FALSO, None, c.pos)
                self.sentencia(a)
                if b is None:
                    self.parchea(si_falso)
                else:
                    al_final = self.emite(Op.SALTA, None, s.pos)
                    self.parchea(si_falso)
                    self.sentencia(b)
                    self.parchea(al_final)
            case Mientras(condicion=c, cuerpo=cuerpo):
                inicio = len(self.codigo.instrucciones)
                self.expresion(c)
                salida = self.emite(Op.SALTA_SI_FALSO, None, c.pos)
                self.sentencia(cuerpo)
                self.emite(Op.SALTA, inicio, s.pos)
                self.parchea(salida)
            case Funcion(nombre=nombre, parametros=ps, cuerpo=cuerpo):
                sub = CompiladorGuanche(nombre, ps)
                for x in cuerpo.sentencias:
                    sub.sentencia(x)
                sub.emite(Op.CONST, sub.constante(None), s.pos)
                sub.emite(Op.DEVUELVE, None, s.pos)
                self.emite(Op.CREA_FUNCION, self.constante(sub.codigo), s.pos)
                self.emite(Op.DECLARA, nombre, s.pos)
            case Devuelve(valor=v):
                if v is None:
                    self.emite(Op.CONST, self.constante(None), s.pos)
                else:
                    self.expresion(v)
                self.emite(Op.DEVUELVE, None, s.pos)


def compilar(programa):
    c = CompiladorGuanche()
    for s in programa.sentencias:
        c.sentencia(s)
    c.emite(Op.FIN)
    return c.codigo


def todos_los_codigos(codigo):
    yield codigo
    for c in codigo.constantes:
        if isinstance(c, Codigo):
            yield from todos_los_codigos(c)


for nombre, fuente in [("FizzBuzz", FIZZBUZZ), ("factorial", FACTORIAL), ("contador", CONTADOR)]:
    codigo = compilar(analizar(fuente))
    for c in todos_los_codigos(codigo):
        saltos = [a for op, a in c.instrucciones if op in SALTOS]
        print(f"{nombre:10} {c.nombre:15} {len(c.instrucciones):3} instrucciones, saltos: {len(saltos)}, constantes: {[mostrar(x) if not isinstance(x, Codigo) else x.nombre for x in c.constantes]}")
        assert all(0 <= a < len(c.instrucciones) for a in saltos)
        assert c.instrucciones[-1][0] in (Op.FIN, Op.DEVUELVE)

FizzBuzz   <programa>       63 instrucciones, saltos: 8, constantes: ['1', '15', '0', 'FizzBuzz', '3', 'Fizz', '5', 'Buzz']
factorial  <programa>       10 instrucciones, saltos: 0, constantes: ['factorial', '5! =', '5']
factorial  factorial        18 instrucciones, saltos: 1, constantes: ['1', 'nulo']
contador   <programa>       15 instrucciones, saltos: 0, constantes: ['crear_contador', '5']
contador   crear_contador    8 instrucciones, saltos: 0, constantes: ['0', 'siguiente', 'nulo']
contador   siguiente         8 instrucciones, saltos: 0, constantes: ['nulo']


## 9. La máquina virtual de pila

Escribir `MaquinaVirtual`, que ejecuta el `Codigo` con una pila de valores y una pila de marcos (código, contador de instrucción y entorno de cada llamada). `LLAMA` apila un marco nuevo en lugar de hacer una llamada recursiva de Python, y `DEVUELVE` lo desapila dejando el resultado en la pila. Reutilizar `Entorno`, `operar` y las nativas del evaluador. Comprobar con 3000 expresiones aleatorias (aritméticas y lógicas) que la máquina da el mismo valor que el evaluador de árbol o el mismo error con el mismo mensaje y la misma posición.

In [10]:
@dataclass(eq=False)
class ClausuraVM:
    codigo: Codigo
    entorno: Entorno

    @property
    def nombre(self):
        return self.codigo.nombre

    @property
    def parametros(self):
        return self.codigo.parametros


@dataclass
class Marco:
    codigo: Codigo
    ip: int
    entorno: Entorno


class MaquinaVirtual:
    def __init__(self, max_marcos=100_000):
        self.salida = []
        self.globales = Entorno()
        self.globales.valores.update(nativas(self.salida))
        self.max_marcos = max_marcos
        self.ejecutadas = 0

    def ejecutar(self, codigo, entorno=None):
        pila = []
        marcos = [Marco(codigo, 0, entorno or self.globales)]
        while True:
            m = marcos[-1]
            op, arg = m.codigo.instrucciones[m.ip]
            pos = m.codigo.posiciones[m.ip]
            m.ip += 1
            self.ejecutadas += 1
            if op in SIMBOLO:
                b = pila.pop()
                pila.append(operar(SIMBOLO[op], pila.pop(), b, pos))
                continue
            match op:
                case Op.CONST:
                    pila.append(m.codigo.constantes[arg])
                case Op.CARGA:
                    pila.append(m.entorno.leer(arg, pos))
                case Op.DECLARA:
                    m.entorno.declarar(arg, pila.pop(), pos)
                case Op.GUARDA:
                    m.entorno.asignar(arg, pila.pop(), pos)
                case Op.SACA:
                    pila.pop()
                case Op.NEG | Op.NO:
                    pila.append(unaria("-" if op == Op.NEG else "no", pila.pop(), pos))
                case Op.SALTA:
                    m.ip = arg
                case Op.SALTA_SI_FALSO:
                    if not logico(pila.pop(), pos):
                        m.ip = arg
                case Op.Y_SALTA | Op.O_SALTA:
                    if logico(pila[-1], pos) == (op == Op.O_SALTA):
                        m.ip = arg
                    else:
                        pila.pop()
                case Op.LOGICO:
                    logico(pila[-1], pos)
                case Op.ENTRA_BLOQUE:
                    m.entorno = Entorno(m.entorno)
                case Op.SALE_BLOQUE:
                    m.entorno = m.entorno.padre
                case Op.CREA_FUNCION:
                    pila.append(ClausuraVM(m.codigo.constantes[arg], m.entorno))
                case Op.LLAMA:
                    args = pila[len(pila) - arg:]
                    del pila[len(pila) - arg:]
                    funcion = pila.pop()
                    comprobar_llamada(funcion, arg, pos)
                    if isinstance(funcion, Nativa):
                        pila.append(llamar_nativa(funcion, args, pos))
                    else:
                        if len(marcos) >= self.max_marcos:
                            raise ErrorGuanche(f"demasiadas llamadas anidadas ({self.max_marcos})", pos)
                        local = Entorno(funcion.entorno)
                        local.valores.update(zip(funcion.parametros, args))
                        marcos.append(Marco(funcion.codigo, 0, local))
                case Op.DEVUELVE:
                    if len(marcos) == 1:
                        raise ErrorGuanche("devuelve fuera de una función", pos)
                    marcos.pop()
                case Op.FIN:
                    return pila[-1] if pila else None


def ejecutar_vm(fuente, optimizar=False):
    arbol = analizar(fuente)
    vm = MaquinaVirtual()
    vm.ejecutar(compilar(plegar(arbol) if optimizar else arbol))
    return vm.salida


def resultado(funcion):
    try:
        return funcion()
    except ErrorGuanche as e:
        return ("error", e.mensaje, e.pos)


print("\n".join(ejecutar_vm(FACTORIAL)))
from collections import Counter

rng = random.Random(9)
errores = Counter()
for _ in range(3000):
    texto_e = expresion_aleatoria(rng, profundidad=3, variables=("a", "b"), logica=rng.random() < 0.5)
    e = analizar_expresion(texto_e)
    variables = {"a": rng.randint(-9, 9), "b": rng.randint(-9, 9)}

    def con_arbol():
        entorno = Entorno()
        entorno.valores.update(variables)
        return Evaluador().evaluar(e, entorno)

    def con_vm():
        entorno = Entorno()
        entorno.valores.update(variables)
        c = CompiladorGuanche()
        c.expresion(e)
        c.emite(Op.FIN)
        return MaquinaVirtual().ejecutar(c.codigo, entorno)

    r1, r2 = resultado(con_arbol), resultado(con_vm)
    assert r1 == r2 and type(r1) is type(r2), (texto_e, r1, r2)
    if isinstance(r1, tuple):
        errores[r1[1]] += 1
print("3000 expresiones con el mismo resultado en ambos motores; errores coincidentes:")
for mensaje, n in errores.most_common(8):
    print(f"  {n:4}  {mensaje}")

5! = 120


3000 expresiones con el mismo resultado en ambos motores; errores coincidentes:
   881  se esperaba un valor lógico y se obtuvo número
   244  división entre cero
    57  no se puede aplicar '-' a lógico
    41  no se puede aplicar '^' a lógico y número
    26  no se puede aplicar '*' a número y lógico
    25  no se puede aplicar '/' a número y lógico
    21  no se puede aplicar '*' a lógico y número
    18  no se puede aplicar '/' a lógico y número


## 10. Desensamblador

Escribir `desensamblar(codigo)`, que muestre el código como hace `dis` en Python: número de instrucción, línea y columna del fuente, nombre de la operación y su argumento interpretado (el valor de la constante, el nombre de la variable o el destino del salto), marcando con `>>` las instrucciones que son destino de un salto y desensamblando después las funciones anidadas. Aplicarlo al factorial y comprobar que hay una línea por instrucción.

In [11]:
def desensamblar(codigo, imprimir=True):
    destinos = {a for op, a in codigo.instrucciones if op in SALTOS}
    lineas = [f"== {codigo.nombre}({', '.join(codigo.parametros)}) =="]
    for i, ((op, arg), pos) in enumerate(zip(codigo.instrucciones, codigo.posiciones)):
        match op:
            case Op.CONST:
                valor = codigo.constantes[arg]
                detalle = f"{arg} ({valor!r})" if isinstance(valor, str) else f"{arg} ({mostrar(valor)})"
            case Op.CREA_FUNCION:
                detalle = f"{arg} (<código {codigo.constantes[arg].nombre}>)"
            case _ if op in SALTOS:
                detalle = f"a {arg}"
            case Op.LLAMA:
                detalle = f"{arg} argumento{'s' * (arg != 1)}"
            case _:
                detalle = "" if arg is None else str(arg)
        marca = ">>" if i in destinos else "  "
        lineas.append(f"{marca} {i:3}  {pos[0]:>2}:{pos[1]:<3} {op.name:<15} {detalle}")
    if imprimir:
        print("\n".join(lineas))
    for c in codigo.constantes:
        if isinstance(c, Codigo):
            if imprimir:
                print()
            lineas += desensamblar(c, imprimir)
    return lineas


lineas = desensamblar(compilar(analizar(FACTORIAL)))
codigos = list(todos_los_codigos(compilar(analizar(FACTORIAL))))
assert len(lineas) == sum(len(c.instrucciones) + 1 for c in codigos)
assert sum(l.startswith(">>") for l in lineas) == len({(id(c), a) for c in codigos for op, a in c.instrucciones if op in SALTOS})

== <programa>() ==
     0   2:1   CREA_FUNCION    0 (<código factorial>)
     1   2:1   DECLARA         factorial
     2   6:1   CARGA           escribe
     3   6:9   CONST           1 ('5! =')
     4   6:17  CARGA           factorial
     5   6:27  CONST           2 (5)
     6   6:26  LLAMA           1 argumento
     7   6:8   LLAMA           2 argumentos
     8   6:1   SACA            
     9   0:0   FIN             

== factorial(n) ==
     0   3:9   CARGA           n
     1   3:14  CONST           0 (1)
     2   3:11  MENOR_IGUAL     
     3   3:11  SALTA_SI_FALSO  a 8
     4   3:17  ENTRA_BLOQUE    
     5   3:28  CONST           0 (1)
     6   3:19  DEVUELVE        
     7   3:17  SALE_BLOQUE     
>>   8   4:14  CARGA           n
     9   4:18  CARGA           factorial
    10   4:28  CARGA           n
    11   4:32  CONST           0 (1)
    12   4:30  RESTA           
    13   4:27  LLAMA           1 argumento
    14   4:16  MULT            
    15   4:5   DEVUELVE        
   

## 11. Comparación con `ast` y `dis` de Python

Tomar la expresión Guanche `(a + 2) * b - 3 ^ 2` y su equivalente en Python. Mostrar el árbol de `ast.parse`, convertirlo al AST de Guanche con `desde_python` y comprobar que coincide con el de nuestro analizador. Después comparar el *bytecode* de CPython (`dis`) con el de la máquina virtual con plegado de constantes: ambos son máquinas de pila, ambos calculan `3 ** 2` al compilar y la secuencia de operaciones debe corresponderse una a una.

In [12]:
import dis

fuente = "(a + 2) * b - 3 ^ 2"
python = a_python(fuente)
arbol_py = ast.parse(python, mode="eval").body
print(ast.dump(arbol_py))

OPERADORES_PY = {ast.Add: "+", ast.Sub: "-", ast.Mult: "*", ast.FloorDiv: "/", ast.Mod: "%", ast.Pow: "^"}


def desde_python(n):
    match n:
        case ast.BinOp(left=a, op=op, right=b):
            return Binaria(OPERADORES_PY[type(op)], desde_python(a), desde_python(b))
        case ast.UnaryOp(op=ast.USub(), operand=x):
            return Unaria("-", desde_python(x))
        case ast.Constant(value=int() as v):
            return Numero(v)
        case ast.Name(id=nombre):
            return Variable(nombre)
    raise ValueError(f"nodo no soportado: {ast.dump(n)}")


assert desde_python(arbol_py) == analizar_expresion(fuente)
print("\nMismo AST en Guanche y en Python\n")
codigo_py = compile(python, "<guanche>", "eval")
dis.dis(codigo_py)
print()
nuestro = compilar_expresion(plegar(analizar_expresion(fuente)))
desensamblar(nuestro)

EQUIVALENCIA = {"LOAD_NAME": "CARGA", "LOAD_CONST": "CONST"}
secuencia_py = [EQUIVALENCIA.get(i.opname) or BINARIAS[i.argrepr.replace("**", "^").replace("//", "/")].name
                for i in dis.get_instructions(codigo_py) if i.opname not in ("RESUME", "RETURN_VALUE")]
secuencia_guanche = [op.name for op, _ in nuestro.instrucciones[:-1]]
print("\nPython: ", secuencia_py)
print("Guanche:", secuencia_guanche)
assert secuencia_py == secuencia_guanche
assert 9 in codigo_py.co_consts and 9 in nuestro.constantes

BinOp(left=BinOp(left=BinOp(left=Name(id='a', ctx=Load()), op=Add(), right=Constant(value=2)), op=Mult(), right=Name(id='b', ctx=Load())), op=Sub(), right=BinOp(left=Constant(value=3), op=Pow(), right=Constant(value=2)))

Mismo AST en Guanche y en Python

  0           RESUME                   0

  1           LOAD_NAME                0 (a)
              LOAD_CONST               0 (2)
              BINARY_OP                0 (+)
              LOAD_NAME                1 (b)
              BINARY_OP                5 (*)
              LOAD_CONST               1 (9)
              BINARY_OP               10 (-)
              RETURN_VALUE

== <expresión>() ==
     0   1:2   CARGA           a
     1   1:6   CONST           0 (2)
     2   1:4   SUMA            
     3   1:11  CARGA           b
     4   1:9   MULT            
     5   1:17  CONST           1 (9)
     6   1:13  RESTA           
     7   0:0   FIN             

Python:  ['CARGA', 'CONST', 'SUMA', 'CARGA', 'MULT', 'CONST', 'RESTA']

## 12. Programas de prueba en ambos motores

Ejecutar en el evaluador de árbol, en la máquina virtual y en la máquina virtual con plegado de constantes una batería de programas (factorial, Fibonacci recursivo frente a iterativo, FizzBuzz, clausuras, recursión mutua y un dibujo con textos) y comprobar que las tres salidas son idénticas entre sí y a la de un programa Python equivalente.

In [13]:
FIBONACCI = """
funcion fib(n) {
    si (n < 2) { devuelve n; }
    devuelve fib(n - 1) + fib(n - 2);
}
funcion fib_iterativo(n) {
    sea a = 0;
    sea b = 1;
    mientras (n > 0) {
        sea siguiente = a + b;
        a = b;
        b = siguiente;
        n = n - 1;
    }
    devuelve a;
}
sea i = 0;
mientras (i <= 15) {
    si (fib(i) != fib_iterativo(i)) { escribe("distintos en", i); }
    i = i + 1;
}
escribe("fib(15) =", fib(15), "y fib(90) =", fib_iterativo(90));
"""
PIRAMIDE = """
funcion repetir(t, n) {
    sea r = "";
    mientras (n > 0) { r = r + t; n = n - 1; }
    devuelve r;
}
sea fila = 1;
mientras (fila <= 4) {
    escribe(repetir(".", 4 - fila) + repetir("*", 2 * fila - 1));
    fila = fila + 1;
}
"""


def fib_python(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


ESPERADO = {
    "factorial": (FACTORIAL, ["5! = 120"]),
    "fibonacci": (FIBONACCI, [f"fib(15) = {fib_python(15)} y fib(90) = {fib_python(90)}"]),
    "fizzbuzz": (FIZZBUZZ, ["FizzBuzz" if i % 15 == 0 else "Fizz" if i % 3 == 0 else "Buzz" if i % 5 == 0 else str(i)
                            for i in range(1, 16)]),
    "contadores": (DOS_CONTADORES, ["10", "100 15 200 20"]),
    "mutua": (MUTUA, ["verdadero verdadero"]),
    "pirámide": (PIRAMIDE, ["." * (4 - f) + "*" * (2 * f - 1) for f in range(1, 5)]),
    "optimizable": (OPTIMIZABLE.rsplit("escribe(1 / 0", 1)[0], ["Gran Canaria 604800"]),
}
print(f"{'programa':12} {'árbol':>6} {'VM':>6} {'VM+plegado':>11}  primera línea")
for nombre, (fuente, esperado) in ESPERADO.items():
    salidas = [ejecutar_arbol(fuente), ejecutar_vm(fuente), ejecutar_vm(fuente, optimizar=True)]
    print(f"{nombre:12} {len(salidas[0]):6} {len(salidas[1]):6} {len(salidas[2]):11}  {salidas[0][0]}")
    assert salidas[0] == salidas[1] == salidas[2] == esperado, nombre
print()
print("\n".join(ejecutar_vm(PIRAMIDE)))

programa      árbol     VM  VM+plegado  primera línea
factorial         1      1           1  5! = 120
fibonacci         1      1           1  fib(15) = 610 y fib(90) = 2880067194370816120
fizzbuzz         15     15          15  1
contadores        2      2           2  10
mutua             1      1           1  verdadero verdadero
pirámide          4      4           4  ...*
optimizable       1      1           1  Gran Canaria 604800

...*
..***
.*****
*******


## 13. Cuánto trabajo hace cada motor

Contar los nodos que visita el evaluador de árbol y las instrucciones que ejecuta la máquina virtual (en lugar de medir tiempos, que varían entre ejecuciones) en tres programas, con y sin plegado de constantes. El programa `CONVERSION` repite en un bucle un cálculo con constantes que el plegado resuelve una sola vez al compilar.

In [14]:
CONVERSION = """
sea i = 0;
sea total = 0;
mientras (i < 500) {
    total = total + i * (60 * 60 * 24) / (1000 * 1000) + (2 ^ 10 - 1024);
    i = i + 1;
}
escribe(total);
"""
print(f"{'programa':12} {'nodos árbol':>12} {'instr. VM':>10} {'VM+plegado':>11} {'ahorro':>7}")
for nombre, fuente in [("fibonacci", FIBONACCI), ("fizzbuzz", FIZZBUZZ), ("conversión", CONVERSION)]:
    arbol = analizar(fuente)
    ev = Evaluador()
    ev.ejecutar(arbol)
    cuentas = []
    for version in (arbol, plegar(arbol)):
        vm = MaquinaVirtual()
        vm.ejecutar(compilar(version))
        assert vm.salida == ev.salida
        cuentas.append(vm.ejecutadas)
    print(f"{nombre:12} {ev.pasos:12} {cuentas[0]:10} {cuentas[1]:11} {1 - cuentas[1] / cuentas[0]:7.0%}")
    if nombre == "conversión":
        assert cuentas[1] < cuentas[0] * 0.75
        print("Total:", ev.salida[0], "=", sum(i * 86400 // 1_000_000 for i in range(500)))
        assert int(ev.salida[0]) == sum(i * 86400 // 1_000_000 for i in range(500))

programa      nodos árbol  instr. VM  VM+plegado  ahorro
fibonacci           81847      86324       86324      0%
fizzbuzz              405        551         551      0%


conversión          13512      15513       10513     32%
Total: 10529 = 10529


## 14. Recursión profunda: la pila de Python frente a una pila propia

El evaluador de árbol usa la recursión de Python para cada llamada de Guanche, así que una recursión de miles de niveles agota el límite de Python. La máquina virtual guarda los marcos en una lista y solo está limitada por `max_marcos`. Comprobarlo con una suma recursiva de 20000 niveles, y ver que una recursión infinita termina con un error controlado en la máquina virtual.

In [15]:
SUMA_RECURSIVA = """
funcion suma(n) {
    si (n == 0) { devuelve 0; }
    devuelve n + suma(n - 1);
}
escribe(suma(20000));
"""
try:
    ejecutar_arbol(SUMA_RECURSIVA)
except RecursionError as e:
    print("Evaluador de árbol:", type(e).__name__, "-", e)

salida = ejecutar_vm(SUMA_RECURSIVA)
print("Máquina virtual:", salida[0])
assert int(salida[0]) == 20000 * 20001 // 2

INFINITA = "funcion f(n) { devuelve f(n + 1); }\nf(0);"
vm = MaquinaVirtual(max_marcos=5000)
try:
    vm.ejecutar(compilar(analizar(INFINITA)))
except ErrorGuanche as e:
    marcar_error(INFINITA, e)
    assert "5000" in e.mensaje

Evaluador de árbol: RecursionError - maximum recursion depth exceeded
Máquina virtual: 200010000
  error 1:26: demasiadas llamadas anidadas (5000)
    funcion f(n) { devuelve f(n + 1); }
                             ^
